# a2_cols

> Phase 21 (`specs/multiagent/phase21/`): the identities and per-board columns of A2's experience screen. A team's 12 honest identities are seated in slots 4-15 by a permutation seeded by team and board (`seating`); `guardIds` and `successIds` draw the four guard identities and the success pair from the team alone, so every arm gets the same roles. `followed` lists a finished board's contests (Phase 19's `contestsOf`) with whether a correct answer by another honest agent followed and how many of ContestLog's watchers then solved; `a2Columns` gives the per-board columns: per identity, guarding, contests, the correct answers that followed, watchers, shares of held clues, answers and turns; the success and failure guards' guarding; the liars' posts on their clues; and the row keys `agents`, `pos` and `seqId` in Phase 19's `rl19_spec` layout.

In [ ]:
#| default_exp a2_cols

In [ ]:
#| hide
from nbdev.showdoc import *

## Identities and seating

Slots 0-3 are the scripted liars. On boards 1-2 the guard identities sit in slots 4-7 (the guard slots of `guardSlots`) in a seeded order and the other 8 in slots 8-15; on boards 3-6 all 12 are permuted. Seeds depend on the team (and the board) only, never on the arm.

In [ ]:
#| export
import json

import numpy as np

from explRL.contest_eval import contestsOf

HONEST, GUARDS, LIARS, BOARDS = 12, 4, 4, 6          # per team of 16 (P21-1)
NAN = float('nan')


def guardIds(team):
    "The team's 4 guard identities (of 0..11), sorted: default_rng([team, 0])."
    return sorted(int(a) for a in np.random.default_rng([team, 0]).choice(HONEST, GUARDS, replace=False))


def successIds(team):
    "The 2 success guards among guardIds(team), sorted: default_rng([team, 99]); the other 2 are the failure guards."
    return sorted(int(a) for a in np.random.default_rng([team, 99]).choice(guardIds(team), 2, replace=False))


def failureIds(team):
    return [a for a in guardIds(team) if a not in successIds(team)]


def seating(team, b):
    "[identity per slot 4..15] for board b: default_rng([team, b]).permutation(12); on boards 1-2 the guard identities first."
    p = [int(a) for a in np.random.default_rng([team, b]).permutation(HONEST)]
    if b <= 2:
        g = set(guardIds(team))
        p = [a for a in p if a in g] + [a for a in p if a not in g]
    return p

## Contests and the correct answers that followed

A contest is Phase 19's (`contestsOf`): an honest post of a clue's true value while the slot shows a false value first posted by a liar. A correct answer followed it when an honest agent other than the contester solved the subtask later (strictly after the contest's turn; only the first correct answer scores, so at most one per subtask). Watchers come from the live `ContestLog` that `a2_seq.runBoard21` attaches to the board (it pays no one): the honest agents who lacked the clue, saw the lie before any true value, and then solved.

In [ ]:
#| export
def followed(board, nLiar):
    """[(t_c, contester, k, j, correctAfter, watchers)] for the board's contests: correctAfter 1 when an honest agent other
    than the contester solved k after t_c; watchers, ContestLog's watchers of that contest who then solved (NaN without a log)."""
    solves = [(t, i, k) for t, i, k, _, res in board.submits if res == 'solved' and i >= nLiar]
    cl = getattr(board, 'contestLog', None)
    w = {(c.t_c, c.who, c.k, c.j): sum(v is not None for v in c.watchers.values()) for c in cl.contests} if cl else {}
    return [(t_c, i, k, j, int(any(kk == k and t > t_c and a != i for t, a, kk in solves)), w.get((t_c, i, k, j), NAN))
            for t_c, i, k, j, _ in contestsOf(board, nLiar)]


def perSlot(board, nLiar, fl=None):
    """Per slot: turns, shares of held clues (any residue, true or not), guarding (held shares / turns), contests, the
    distinct contested subtasks a correct answer followed, watchers, answers."""
    N, fl = board.nAgents, followed(board, nLiar) if fl is None else fl
    z = lambda: [0] * N
    turns, held, cont, fol, wat, ans = z(), z(), z(), z(), [0.0] * N, z()
    for r in board.ledger:
        turns[r[0]] += 1
    for s in board.shareLog:
        held[s[1]] += s[1] in board.tasks[s[2]].holders[s[3]]
    for s in board.submits:
        ans[s[1]] += 1
    for t_c, i, k, j, ca, w in fl:
        cont[i], wat[i] = cont[i] + 1, wat[i] + w
    for i, k in {(c[1], c[2]) for c in fl if c[4]}:
        fol[i] += 1
    guard = [h / n if n else NAN for h, n in zip(held, turns)]
    return dict(turns=turns, heldShares=held, guarding=guard, contests=cont, followed=fol, watchers=wat, answers=ans)

## Per-board columns

The world of an A2 cell carries `seat` (the identities in slots 4-15), `team` and `board`; a P2 board has none of them, and its identities are its slots. Per-identity columns are JSON lists ordered by identity (0..11), so a team's boards line up whatever the seating.

In [ ]:
#| export
def a2Columns(cfg, board):
    """Phase 21's columns for one board: per identity (JSON, identity order) guarding, contests, followed, watchers, held
    shares, answers, turns; the success and failure guards' mean guarding; the liars' false posts on clues a success or a
    failure guard holds; the row keys agents (per slot: the liar's slot, or identity + 4), pos (board - 1) and seqId (team)."""
    w, nL, N = cfg.world, cfg.nLiar, cfg.N
    seat = [int(a) for a in (w.get('seat') or range(N - nL))]
    slotOf = {a: nL + s for s, a in enumerate(seat)}
    per = perSlot(board, nL)
    byId = lambda xs: json.dumps([xs[slotOf[a]] for a in sorted(slotOf)])
    out = {f'id{k[0].upper()}{k[1:]}': byId(v) for k, v in per.items()}
    out['agents'] = json.dumps(list(range(nL)) + [a + nL for a in seat])
    if 'team' in w:
        team, mean = int(w['team']), lambda ids: float(np.nanmean([per['guarding'][slotOf[a]] for a in ids]))
        S, F = {slotOf[a] for a in successIds(team)}, {slotOf[a] for a in failureIds(team)}
        lies = [s for s in board.shareLog if s[1] < nL and s[6]]
        on = lambda slots: sum(bool(slots & set(board.tasks[s[2]].holders[s[3]])) for s in lies)
        out.update(succGuarding=mean(successIds(team)), failGuarding=mean(failureIds(team)), liarLies=len(lies),
                   liarOnSuccess=on(S), liarOnFailure=on(F), pos=int(w['board']) - 1, seqId=team)
    return out

## Tests

In [ ]:
# test_seating (AC-5, FR-B1): seats, guard identities and the success pair depend on the team (and board) only; the
# guard identities sit in slots 4-7 on boards 1-2; every seating is a permutation of the 12 identities
from types import SimpleNamespace
from explRL.a2_cols import *

for team in (2100, 2101, 2107, 2190):
    g, s = guardIds(team), successIds(team)
    assert len(g) == 4 == len(set(g)) and set(s) < set(g) and len(s) == 2 and sorted(failureIds(team) + s) == g
    for b in range(1, 7):
        p = seating(team, b)
        assert sorted(p) == list(range(12)) and p == seating(team, b)
        if b <= 2:
            assert set(p[:4]) == set(g)
assert len({tuple(guardIds(t)) for t in range(2100, 2108)}) > 1 and seating(2100, 3) != seating(2100, 4)

In [ ]:
# followed and perSlot on a hand-built board (AC-6's counts): liar 0 lies on clue (k, 0); agent 2 (not a holder) opens k
# after the lie, so it is a watcher; holder 3 contests; agent 2 then solves k: one correct answer followed, one watcher.
# Agent 4 solving instead also follows (and watched); a board where the contester solves has nothing followed.
from explRL.clueboard import ClueBoardEnv
from explRL.contest import ContestLog

def handBoard(solver):
    b = ClueBoardEnv(5, 6, m=2, r=1, R=30)
    b.contestLog = ContestLog(b, range(1), ())
    k = b.open[0]
    assert [set(h) for h in b.tasks[k].holders] == [{3}, {1}] and b.tasks[k].task.check(29)
    b.apply(0, ('share', (k, 0, 2)), 0.0)                     # the lie (true residue 4)
    b.apply(2, ('open', k), 0.0)
    b.apply(solver, ('open', k), 0.5) if solver != 2 else None
    b.apply(3, ('share', (k, 0, 4)), 1.0)                     # the contest
    b.apply(1, ('share', (k, 1, 7)), 1.0)                     # a true post with no lie on its slot: no contest
    b.apply(solver, ('submit', 29), 2.0)
    return b, k

b, k = handBoard(2)
fl = followed(b, 1)
assert fl == [(1.0, 3, k, 0, 1, 1)], fl
ps = perSlot(b, 1, fl)
assert ps['contests'] == [0, 0, 0, 1, 0, 0] and ps['followed'] == [0, 0, 0, 1, 0, 0] and ps['watchers'][3] == 1
assert ps['heldShares'] == [0, 1, 0, 1, 0, 0] and ps['guarding'][3] == 1.0 and ps['guarding'][1] == 1.0 and ps['answers'][2] == 1
cfg = SimpleNamespace(world={}, nLiar=1, N=6)
cols = a2Columns(cfg, b)
assert json.loads(cols['agents']) == [0, 1, 2, 3, 4, 5] and json.loads(cols['idContests']) == [0, 0, 1, 0, 0]
assert 'succGuarding' not in cols                                # no team: a P2-like board
b3, k3 = handBoard(4)                                            # agent 4 solves: still a follower (not the contester)
assert followed(b3, 1) == [(1.0, 3, k3, 0, 1, 1)]               # 4 opened k after the lie, before the contest: a watcher
b4 = ClueBoardEnv(5, 6, m=2, r=1, R=30)
k4 = b4.open[0]
b4.apply(0, ('share', (k4, 0, 2)), 0.0); b4.apply(3, ('open', k4), 0.0); b4.apply(3, ('share', (k4, 0, 4)), 1.0)
b4.apply(3, ('share', (k4, 1, 7)), 1.5); b4.apply(3, ('submit', 29), 2.0)
f = followed(b4, 1)
assert len(f) == 1 and f[0][:5] == (1.0, 3, k4, 0, 0) and f[0][5] != f[0][5]   # the contester solved: none followed; no log: NaN

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()